# TFM - Preparación de Datos y Z-Scores

Ccódigo utilizado para preparar el dataset antes de introducirlo en los modelos de Machine Learning (K-Means y DBSCAN).

En este bloque se definen los 30 activos del estudio (15 acciones y 15 ETFs), se descarga el histórico completo desde Yahoo Finance y se limpian los días festivos. Esto evita la aparición de volúmenes a cero que puedan generar errores en los algoritmos.

Posteriormente, se calculan los indicadores y se pasa todo a Z-Scores móviles de 1 año. 

In [15]:
# =============================================================================
# BLOQUE 0 — PARÁMETROS GLOBALES
# =============================================================================

import yfinance as yf
import pandas as pd
import numpy as np
import ta
import os
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

TICKERS_TFM = {
    "spain": ["SAN.MC", "FER.MC", "ACS.MC", "ITX.MC", "REP.MC", "AMS.MC"],
    "usa"  : ["AAPL", "MSFT", "NVDA", "JPM", "BLK", "TSLA", "DPZ", "CAT", "XOM"],
    "etfs" : ["XLK", "SMH", "XLV", "XLI", "XLY", "XLP", "GLD", "SPY", 
              "QQQ", "EWP", "MTUM", "XLE", "XLF", "TLT", "XLU"]
}
ALL_TICKERS = [t for sublist in TICKERS_TFM.values() for t in sublist]

FECHA_INICIO_ANALISIS = "2017-01-01"
#FECHA_INICIO_ANALISIS = "2020-01-01"

# Ventanas de indicadores
W_RSI      = 14
W_MACD_S   = 12
W_MACD_L   = 26
W_MACD_SIG = 9
W_ATR      = 14
W_VOL_REAL = 21
W_VOL_ROC  = 5
W_ZSCORE   = 252   # 1 año bursátil

# Rutas de salida
OUTPUT_DIR_RAW = "data/raw"
OUTPUT_DIR_PROCESSED = "data/processed"
FILE_BRUTO = os.path.join(OUTPUT_DIR_RAW, "01_dataset_bruto.csv")
FILE_CLEAN = os.path.join(OUTPUT_DIR_PROCESSED, "02_dataset_final.csv")

# --- LOOKBACK BUFFER ---
WARMUP_BIZ_DAYS = W_ZSCORE + 35
WARMUP_CAL_DAYS = int(np.ceil(WARMUP_BIZ_DAYS * (365 / 252))) + 30

FECHA_DESCARGA = (
    pd.Timestamp(FECHA_INICIO_ANALISIS) - pd.Timedelta(days=WARMUP_CAL_DAYS)
).strftime("%Y-%m-%d")

print(f"  Análisis desde : {FECHA_INICIO_ANALISIS}")
print(f"  Descarga desde : {FECHA_DESCARGA}  (buffer de {WARMUP_CAL_DAYS} días naturales)")

  Análisis desde : 2017-01-01
  Descarga desde : 2015-10-13  (buffer de 446 días naturales)


### 1. Descarga y unificación de datos
En este primer paso se realiza la conexión a la API. Tras la descarga, se manipulan las tablas (usando *stack* y *merge*) para dejar la base de datos en formato largo (*long format*). De esta forma, cada fila representa la estructura "Fecha - Ticker - Dato", lo cual simplifica la agrupación y las operaciones posteriores con Pandas.

In [ ]:
# =============================================================================
# BLOQUE 1 — DESCARGA DE DATOS
# =============================================================================

def descargar_datos(tickers: list):
    raw = yf.download(
        tickers,
        start=FECHA_DESCARGA,
        interval="1d",
        auto_adjust=True,
        progress=False,
    )
    return raw["Close"], raw["High"], raw["Low"], raw["Volume"]

# =============================================================================
# BLOQUE 2 — UNIFICACIÓN DE SERIES TEMPORALES
# =============================================================================

def unificar_series_temporales(close_df, high_df, low_df, volume_df):

    def _alinear_datos(df_input, nombre_col):
        df_temp = df_input.stack().reset_index()
        df_temp.columns = ["Fecha", "Ticker", nombre_col]
        return df_temp

    # Se unen todas las fuentes (Precio, High, Low, Volumen) en una sola tabla
    df = _alinear_datos(close_df,   "Precio_Adj")
    df = df.merge(_alinear_datos(high_df,   "High"),    on=["Fecha", "Ticker"])
    df = df.merge(_alinear_datos(low_df,    "Low"),     on=["Fecha", "Ticker"])
    df = df.merge(_alinear_datos(volume_df, "Volumen"), on=["Fecha", "Ticker"])

    # Normalización de fechas y ordenación
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df = df.sort_values(["Ticker", "Fecha"]).reset_index(drop=True)

    return df

### 2. Limpieza de festivos y cálculo de variables
Tratamiento de los días en los que la bolsa cierra. A veces, la API registra la fecha dejándola en blanco o con volumen 0. Aquí se aplica un filtro para esos casos y, si falta algún precio puntual, se copia el del día anterior (con un límite de 3 días seguidos; si el hueco es mayor, se elimina la fila para no inventar datos).

Una vez que los datos están limpios, se calculan las variables que se usarán en el modelo de clustering: retornos logarítmicos, RSI, MACD, ATR, volatilidades, etc.

In [11]:
# =============================================================================
# BLOQUE 3 — LIMPIEZA + ELIMINAR FESTIVOS
# =============================================================================

def limpiar_y_eliminar_festivos(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Rellenar nulos y crear el filtro de festivos
    df["Volumen"] = df["Volumen"].fillna(0)
    filtro_festivos = (df["Volumen"] == 0)

    total_festivos = filtro_festivos.sum()
    print(f"    -> Días festivos (Volumen 0 o vacío) detectados y eliminados: {total_festivos:,}")

    # Aplicar filtro
    df = df[~filtro_festivos].copy()

    nans_precio_antes = df["Precio_Adj"].isna().sum()

    # Revisión para pequeños fallos de la API (Límite 3 días)
    for col in ["Precio_Adj", "High", "Low"]:
        df[col] = df.groupby("Ticker")[col].transform(lambda s: s.ffill(limit=3))

    nans_precio_despues = df["Precio_Adj"].isna().sum()
    salvados = nans_precio_antes - nans_precio_despues

    if nans_precio_antes > 0:
        print(f"    -> Precios en blanco detectados: {nans_precio_antes:,}")
        print(f"    -> Precios salvados copiando el día anterior: {salvados:,}")
    else:
        print("    -> No se detectaron precios en blanco tras la eliminación de festivos.")

    filas_insalvables = df[df[["Precio_Adj", "High", "Low"]].isna().any(axis=1)]

    if not filas_insalvables.empty:
        resumen_borrados = filas_insalvables.groupby("Ticker").size()
        print(f"    -> ADVERTENCIA: Se borrarán definitivamente {nans_precio_despues:,} registros (huecos > 3 días):")
        for ticker, cantidad in resumen_borrados.items():
            print(f"       * {ticker}: {cantidad:,} días eliminados")

    df = df.dropna(subset=["Precio_Adj", "High", "Low"])
    return df

# =============================================================================
# BLOQUE 4 — INDICADORES
# =============================================================================

def calcular_indicadores(df: pd.DataFrame) -> pd.DataFrame:
    def _aplicar_formulas(grupo: pd.DataFrame) -> pd.DataFrame:
        grupo["DailyLogReturn"] = np.log(grupo["Precio_Adj"] / grupo["Precio_Adj"].shift(1))
        grupo["ALR1W"]          = grupo["DailyLogReturn"].rolling(5).sum()
        grupo["ALR1M"]          = grupo["DailyLogReturn"].rolling(21).sum()

        grupo["RSI_14"]   = ta.momentum.rsi(grupo["Precio_Adj"], window=W_RSI)
        grupo["MACD_Hist"] = ta.trend.macd_diff(
            grupo["Precio_Adj"], window_slow=W_MACD_L, window_fast=W_MACD_S, window_sign=W_MACD_SIG,
        )

        grupo["Volatilidad_21d"] = grupo["DailyLogReturn"].rolling(W_VOL_REAL).std()
        grupo["ATR_14"] = ta.volatility.average_true_range(
            high=grupo["High"], low=grupo["Low"], close=grupo["Precio_Adj"], window=W_ATR,
        )

        log_vol         = np.log1p(grupo["Volumen"])
        grupo["Volumen_ROC"] = ta.momentum.roc(log_vol, window=W_VOL_ROC)

        vol_media_20 = grupo["Volumen"].rolling(20).mean()
        grupo["Vol_Relativo"] = grupo["Volumen"] / vol_media_20.replace(0, np.nan)

        return grupo

    df = df.groupby("Ticker", group_keys=False).apply(_aplicar_formulas)
    return df

### 3. Z-Score y exportación
Finalmente, se aplica el Z-Score con una ventana de 252 días (equivalente a un año bursátil). Como al principio de la serie se generan valores nulos obligatoriamente (se necesitan días previos para poder calcular la media y la desviación de la ventana), se recorta ese periodo inicial de "warmup" y se guarda el CSV final únicamente a partir de la fecha de inicio oficial.

In [ ]:
# =============================================================================
# BLOQUE 5 — Z-SCORE ROLLING POR TICKER
# =============================================================================

def estandarizar_zscore_rolling(df: pd.DataFrame) -> pd.DataFrame:
    columnas_Z = ["DailyLogReturn", "ALR1W", "ALR1M", "MACD_Hist",
                  "Volatilidad_21d", "ATR_14", "Volumen_ROC", "RSI_14", "Vol_Relativo"]

    def _zscore(series: pd.Series, w=W_ZSCORE) -> pd.Series:
        mu  = series.rolling(w, min_periods=w // 2).mean()
        std = series.rolling(w, min_periods=w // 2).std()
        return (series - mu) / std.replace(0, np.nan)

    def _apply(g):
        for col in columnas_Z:
            if col in g.columns:
                g[f"{col}_Z"] = _zscore(g[col])
        return g

    df = df.groupby("Ticker", group_keys=False).apply(_apply)
    return df

# =============================================================================
# BLOQUE 6 — RECORTE Y LIMPIEZA FINAL
# =============================================================================

def recortar(df: pd.DataFrame) -> pd.DataFrame:
    df = df[df["Fecha"] >= FECHA_INICIO_ANALISIS].copy()

    variables_ml = [c for c in df.columns if c.endswith("_Z")]
    df = df.dropna(subset=variables_ml)

    # Mantenemos Fecha, Ticker y las columnas Z
    columnas_finales = ["Fecha", "Ticker"] + variables_ml
    df = df[columnas_finales]

    return df

# =============================================================================
# EXPORTACIÓN
# =============================================================================

def exportar(df_bruto, df_clean):
    os.makedirs(OUTPUT_DIR_RAW, exist_ok=True)
    os.makedirs(OUTPUT_DIR_PROCESSED, exist_ok=True)
    df_bruto.to_csv(FILE_BRUTO, index=False, sep=";", decimal=",")
    df_clean.to_csv(FILE_CLEAN, index=False, sep=";", decimal=",")

    print(f"\n{'='*55}")
    print(f"   Bruto    → {FILE_BRUTO}   [{len(df_bruto):,} filas]")
    print(f"   Final    → {FILE_CLEAN}   [{len(df_clean):,} filas]")

    # Comprobación final de nulos
    nulos_finales = df_clean.isna().sum().sum()
    print(f"   Valores nulos totales: {nulos_finales}")
    print(f"{'='*55}")

# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    print("=" * 55)
    print(" DATASET ")
    print("=" * 55)

    close_df, high_df, low_df, volume_df = descargar_datos(ALL_TICKERS)
    df = unificar_series_temporales(close_df, high_df, low_df, volume_df)

    df = limpiar_y_eliminar_festivos(df)
    df = calcular_indicadores(df)
    df_features = df.copy()

    df = estandarizar_zscore_rolling(df)
    df_clean = recortar(df)

    exportar(df_features, df_clean)

 DATASET 
### Check 1 estructura
<class 'pandas.DataFrame'>
MultiIndex([( 'Close',   'AAPL'),
            ( 'Close', 'ACS.MC'),
            ( 'Close', 'AMS.MC'),
            ( 'Close',    'BLK'),
            ( 'Close',    'CAT'),
            ( 'Close',    'DPZ'),
            ( 'Close',    'EWP'),
            ( 'Close', 'FER.MC'),
            ( 'Close',    'GLD'),
            ( 'Close', 'ITX.MC'),
            ...
            ('Volume',   'TSLA'),
            ('Volume',    'XLE'),
            ('Volume',    'XLF'),
            ('Volume',    'XLI'),
            ('Volume',    'XLK'),
            ('Volume',    'XLP'),
            ('Volume',    'XLU'),
            ('Volume',    'XLV'),
            ('Volume',    'XLY'),
            ('Volume',    'XOM')],
           names=['Price', 'Ticker'], length=150)
### Check 2 shapes 
(2723, 30)
(2723, 30)
(2723, 30)
(2723, 30)
### Check 3 nombres columnas
['AAPL', 'ACS.MC', 'AMS.MC', 'BLK', 'CAT', 'DPZ', 'EWP', 'FER.MC', 'GLD', 'ITX.MC', 'JPM', 'MSFT', '

##  (Resumen de los 15 ETFs)


- `SPY` (S&P 500), `QQQ` (Nasdaq 100), `EWP` (Bolsa de España).
- `MTUM` (ETF oficial de la estrategia de Momentum) 
- `XLK` (Tecnología), `SMH` (Microchips), `XLF` (Finanzas), `XLE` (Energía), `XLI` (Industria), `XLY` (Consumo cíclico).
- `XLP` (Consumo básico), `XLV` (Salud), `XLU` (Servicios públicos/Utilities).
- `GLD` (Oro físico) y `TLT` (Bonos del Tesoro a largo plazo). 